# 4단계 평가 — Run3 모델을 test셋으로 채점

**목표**: 3단계에서 선정한 baseline(Run3: lr=2e-5, epoch=3)을 한 번도 보지 않은 test셋(9,026건)으로 채점하고, 오분류 패턴을 분석한다.

실행 스크립트: `scripts/evaluate.py`
```bash
python scripts/evaluate.py --model-dir results/run3/best_model --test-file data/processed/test.jsonl
```


## 1. test set 성능

In [ ]:
import json
metrics = json.load(open("../results/run3/test_metrics.json", encoding="utf-8"))
print(json.dumps(metrics, indent=2))
print()
print(open("../results/run3/test_classification_report.txt", encoding="utf-8").read())

{
  "test_f1": 0.9045
}

              precision    recall  f1-score   support

         LOC       0.76      0.89      0.82       291
         ORG       0.90      0.94      0.92      6573
         PER       0.61      0.50      0.55       269

   micro avg       0.89      0.92      0.90      7133
   macro avg       0.76      0.78      0.76      7133
weighted avg       0.89      0.92      0.90      7133


val F1(0.9077)과 test F1(0.9045)이 거의 같다 → **val에 과적합되지 않고 그대로 일반화됐다**는 뜻. 목표(0.80)를 확실히 초과.

## 2. 오분류 사례 분석 (상위 200건)

In [ ]:
import json
from collections import Counter

mistakes = json.load(open("../results/run3/test_misclassified.json", encoding="utf-8"))
print("collected:", len(mistakes))
print(Counter((m["kind"], m["type"]) for m in mistakes))

collected: 200
Counter({('false_positive (오탐)', 'ORG'): 96, ('false_negative (놓침)', 'ORG'): 72, ('false_positive (오탐)', 'LOC'): 9, ('false_negative (놓침)', 'LOC'): 9, ('false_positive (오탐)', 'PER'): 8, ('false_negative (놓침)', 'PER'): 6})


전체 오류의 84%(168/200)가 ORG. 실제 사례를 읽어보면 세 가지 패턴이 뚜렷하다:

1. **중첩된 기관명의 경계 오류** (가장 흔함): `국가정보원 심리전단` → `국가정보원`만 잡거나, `연세대학교 의과대학 세브란스병원` → 뒤에 `소아 비뇨기과`까지 과하게 넓혀서 잡음.
2. **"트위터"를 한 번도 못 잡음**: 학습 데이터에 SNS/플랫폼명이 ORG로 등장하는 빈도가 낮아서 일반적인 기관명 패턴과 다른 이런 이름을 학습하지 못함.
3. **정답 라벨 자체의 노이즈**: `다음과 같은 사실들이`의 `다음`이 정답에 ORG로 라벨링된 사례 발견 — 실제로는 "아래"라는 뜻의 일반 단어. 원본 데이터의 규칙 기반 라벨링 자체의 오류로 보임.

PER·LOC 오류(각 14~17건)도 대부분 비슷한 **경계 오류** 패턴(예: `아산시 선장면장`에서 직책 접미사 `-장` 포함 여부, `#이름#` placeholder를 `#`과 `이름`으로 잘못 나눠 잡음)이었다.

In [ ]:
for m in mistakes[:10]:
    print(m["kind"], m["type"], "|", m["span_text"], "| context:", m["context"])

false_negative (놓침) ORG | 연세대학교 의과대학 세브란스병원 | context: 그 후 신청인은 연세대학교 의과대학 세브란스병원 소아 비뇨기과에서
false_positive (오탐) ORG | 연세대학교 의과대학 세브란스병원 소아 비뇨기과 | context: 그 후 신청인은 연세대학교 의과대학 세브란스병원 소아 비뇨기과에서 사건본인에
false_negative (놓침) ORG | 광주지방법원 본원 합의부 | context: , 그 부분 사건을 광주지방법원 본원 합의부에 환송한다. 나머지
false_positive (오탐) ORG | 광주지방법원 | context: , 그 부분 사건을 광주지방법원 본원 합의부에 환송
false_positive (오탐) LOC | 아산시 선장면 | context: 해 11. 27. 아산시 선장면장에 신고 ) 은
false_positive (오탐) PER | # 이름 | context: ##양시로 가서 # 이름 # 와 만났으나 #
false_negative (놓침) ORG | 국가정보원 심리전단 | context: 12. 경까지 국가정보원 심리전단장으로 근무하였
false_positive (오탐) ORG | 국가정보원 | context: 12. 경까지 국가정보원 심리전단장으로 근무
false_positive (오탐) ORG | 합동참모본부 | context: 4. 경까지 합동참모본부 민군심리전
false_negative (놓침) ORG | 트위터 | context: ##정보원 심리전단이 트위터에서도 사이버 활동을


## 3. 공식 벤치마크와 비교

- 공식 GPT-OSS-120B(QLoRA): F1 **0.9589** (AI Hub 비공개 test셋)
- 본 프로젝트 KLUE-BERT(Run3): F1 **0.9045** (자체 분리한 test셋)

차이(0.0544)의 원인: (1) 모델 크기 110M vs 120B, (2) test셋이 다름(직접 비교는 참고용), (3) 위에서 확인한 중첩 기관명 경계 오류가 실제 F1을 깎는 가장 큰 요인으로 보임 — 대형 모델일수록 더 긴 문맥을 반영해 경계를 잘 잡을 가능성이 있으나 직접 검증은 못 했다.

## 결론 및 다음 단계

1. Run3 모델이 test셋에서 F1 0.9045로 목표(0.80)를 확실히 초과, val과 거의 동일해 과적합 없음.
2. 약점은 뚜렷하다: **중첩 기관명 경계 오류(ORG 오류의 대다수)**, **PER 인식 정확도(F1 0.55)**, **학습 데이터에 드문 유형(트위터 등) 미탐지**.
3. 원본 데이터 라벨링 자체의 노이즈도 일부 발견(`다음`이 ORG로 잘못 라벨링) — 완벽한 정답셋이 아니라는 점을 감안해야 함.
4. 다음: **5단계 Streamlit 데모 앱** — Run3 모델로 실시간 익명화 데모 구현.